In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# Upload file
from google.colab import files

uploaded = files.upload()

FILE_PATH = list(uploaded.keys())[0]

print(f"Uploaded file: {FILE_PATH}")

OUTPUT_DIR = "data_cleaned"
REPORT_DIR = "data_quality_reports"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("=" * 80)
print("DA18 - LEVEL 1: DATA CLEANING & QUALITY ASSESSMENT")
print("=" * 80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

def load_all_sheets(path):

    print(f"\nLoading from: {path}")

    sheets = {}

    try:

        excel_file = pd.ExcelFile(path)

        print("\nSheets found:")
        print(excel_file.sheet_names)

        for sheet in excel_file.sheet_names:

            sheets[sheet] = pd.read_excel(
                path,
                sheet_name=sheet
            )

            print(
                f"✓ {sheet}: "
                f"{sheets[sheet].shape[0]:,} rows × "
                f"{sheets[sheet].shape[1]} cols"
            )

    except Exception as e:

        print("ERROR:", e)
        return None

    return sheets
raw_data = load_all_sheets(FILE_PATH)

if raw_data is None:
    raise Exception("Cannot load file")

print("\nAvailable tables:")
print(list(raw_data.keys()))
employees = raw_data.get(
    "employees",
    pd.DataFrame()
)

stores = raw_data.get(
    "stores",
    pd.DataFrame()
)

monthly_perf = raw_data.get(
    "monthly_performance",
    pd.DataFrame()
)

role_kpis = raw_data.get(
    "role_kpis",
    pd.DataFrame()
)

business_outcomes = raw_data.get(
    "business_outcomes",
    pd.DataFrame()
)
print("\nTable Sizes")

print("employees:", employees.shape)
print("stores:", stores.shape)
print("monthly_performance:", monthly_perf.shape)
print("role_kpis:", role_kpis.shape)
print("business_outcomes:", business_outcomes.shape)

def quality_check(df, name):

    print("\n" + "=" * 60)
    print(name.upper())
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicates:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)
for name, table in {

    "employees": employees,
    "stores": stores,
    "monthly_performance": monthly_perf,
    "role_kpis": role_kpis,
    "business_outcomes": business_outcomes

}.items():

    quality_check(table, name)
def parse_dates(df, cols):

    for col in cols:

        if col in df.columns:

            df[col] = pd.to_datetime(
                df[col],
                dayfirst=True,
                errors="coerce"
            )

    return df
employees = parse_dates(
    employees,
    ["Hire_Date", "Exit_Date"]
)

stores = parse_dates(
    stores,
    ["Opening_Date"]
)

employees = employees.drop_duplicates()
stores = stores.drop_duplicates()
monthly_perf = monthly_perf.drop_duplicates()
role_kpis = role_kpis.drop_duplicates()
business_outcomes = business_outcomes.drop_duplicates()

# Export cleaned data to Excel

excel_path = os.path.join(
    OUTPUT_DIR,
    "Employee_Performance_Cleaned.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    employees.to_excel(
        writer,
        sheet_name="employees",
        index=False
    )

    stores.to_excel(
        writer,
        sheet_name="stores",
        index=False
    )

    monthly_perf.to_excel(
        writer,
        sheet_name="monthly_performance",
        index=False
    )

    role_kpis.to_excel(
        writer,
        sheet_name="role_kpis",
        index=False
    )

    business_outcomes.to_excel(
        writer,
        sheet_name="business_outcomes",
        index=False
    )

print("Cleaned file saved:")
print(excel_path)
from google.colab import files

files.download(excel_path)

Saving Employee Performance Dataset.xlsx to Employee Performance Dataset.xlsx
Uploaded file: Employee Performance Dataset.xlsx
DA18 - LEVEL 1: DATA CLEANING & QUALITY ASSESSMENT
Start time: 2026-06-12 15:39:15

Loading from: Employee Performance Dataset.xlsx

Sheets found:
['employees', 'stores', 'monthly_performance', 'role_kpis', 'business_outcomes', 'Data_Dictionary']
✓ employees: 7,500 rows × 18 cols
✓ stores: 150 rows × 7 cols
✓ monthly_performance: 236,591 rows × 13 cols
✓ role_kpis: 236,591 rows × 9 cols
✓ business_outcomes: 16,200 rows × 9 cols
✓ Data_Dictionary: 69 rows × 5 cols

Available tables:
['employees', 'stores', 'monthly_performance', 'role_kpis', 'business_outcomes', 'Data_Dictionary']

Table Sizes
employees: (7500, 18)
stores: (150, 7)
monthly_performance: (236591, 13)
role_kpis: (236591, 9)
business_outcomes: (16200, 9)

EMPLOYEES
Shape: (7500, 18)

Missing Values:
Employee_Id                    0
Full_Name                      0
Age                            0
Ed

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# =====================================================
# MACHINE LEARNING REGRESSION
# Predict Productivity Index
# Employee_Performance_Cleaned.xlsx
# =====================================================

!pip install xgboost openpyxl -q

import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

# =====================================================
# UPLOAD FILE
# =====================================================

uploaded = files.upload()

FILE_PATH = next(iter(uploaded))

print(f"Uploaded File: {FILE_PATH}")

# =====================================================
# LOAD DATA
# =====================================================

employees = pd.read_excel(
    FILE_PATH,
    sheet_name="employees"
)

stores = pd.read_excel(
    FILE_PATH,
    sheet_name="stores"
)

performance = pd.read_excel(
    FILE_PATH,
    sheet_name="monthly_performance"
)

kpis = pd.read_excel(
    FILE_PATH,
    sheet_name="role_kpis"
)

business = pd.read_excel(
    FILE_PATH,
    sheet_name="business_outcomes"
)

# =====================================================
# REMOVE DUPLICATES IN SOURCE TABLES
# =====================================================

employees = employees.drop_duplicates()
stores = stores.drop_duplicates()
performance = performance.drop_duplicates()
kpis = kpis.drop_duplicates()
business = business.drop_duplicates()

print("\nAfter Source Deduplication")

print("employees :", employees.shape)
print("stores :", stores.shape)
print("performance :", performance.shape)
print("kpis :", kpis.shape)
print("business :", business.shape)

# =====================================================
# PREVENT MANY TO MANY JOIN
# =====================================================

business = business.groupby(
    [
        "Store_Id",
        "Department",
        "Year_Month"
    ],
    as_index=False
).agg({

    "Customer_Satisfaction":"mean",
    "Nps_Score":"mean",
    "Waste_Percentage":"mean",
    "On_Time_Delivery":"mean"

})

print("\nBusiness After Aggregation")

print(business.shape)

# =====================================================
# MERGE DATA
# =====================================================

df = performance.merge(
    employees,
    on="Employee_Id",
    how="left"
)

df = df.merge(
    stores,
    on="Store_Id",
    how="left"
)

df = df.merge(
    kpis[
        [
            "Employee_Id",
            "Year_Month",
            "Productivity_Index"
        ]
    ],
    on=[
        "Employee_Id",
        "Year_Month"
    ],
    how="left"
)

df = df.merge(
    business,
    on=[
        "Store_Id",
        "Department",
        "Year_Month"
    ],
    how="left"
)

print("\nShape Before Dedup")
print(df.shape)

df = df.drop_duplicates()

print("\nShape After Dedup")
print(df.shape)

# =====================================================
# FEATURES
# =====================================================

features = [

    "Training_Hours",
    "Overtime_Hours",
    "Absenteeism_Days",
    "Monthly_Bonus",
    "Employee_Satisfaction",
    "Engagement_Index",
    "Manager_Evaluation",

    "Customer_Satisfaction",
    "Nps_Score",
    "Waste_Percentage",
    "On_Time_Delivery",

    "Job_Level",
    "Department",
    "Employment_Type",
    "Education_Level",
    "Store_Type"
]

target = "Productivity_Index"

# =====================================================
# NUMERIC
# =====================================================

numeric_features = [

    "Training_Hours",
    "Overtime_Hours",
    "Absenteeism_Days",
    "Monthly_Bonus",
    "Employee_Satisfaction",
    "Engagement_Index",
    "Manager_Evaluation",

    "Customer_Satisfaction",
    "Nps_Score",
    "Waste_Percentage",
    "On_Time_Delivery"
]

# =====================================================
# CATEGORICAL
# =====================================================

categorical_features = [

    "Job_Level",
    "Department",
    "Employment_Type",
    "Education_Level",
    "Store_Type"
]

# =====================================================
# REMOVE NULL TARGET
# =====================================================

df = df.dropna(subset=[target])

# =====================================================
# CORRELATION CHECK
# =====================================================

print("\nTop Correlations")

corr_check = (
    df.select_dtypes(include=np.number)
      .corr()["Productivity_Index"]
      .sort_values(ascending=False)
)

print(corr_check.head(15))

# =====================================================
# FINAL DEDUP
# =====================================================

before_rows = len(df)

df = df.drop_duplicates()

after_rows = len(df)

print(
    f"\nRemoved {before_rows-after_rows} duplicate rows"
)

print("Final Shape:", df.shape)

# =====================================================
# X / y
# =====================================================
print("\nCorrelation After Removing Leakage Variable")

corr_check = (
    df[
        numeric_features + [target]
    ]
    .corr()[target]
    .sort_values(ascending=False)
)

print(corr_check)

X = df[features]
y = df[target]

# =====================================================
# PREPROCESSING
# =====================================================

numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    [
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================================
# EVALUATION FUNCTION
# =====================================================

def evaluate_model(model, name):

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print("MAE :", round(mae,4))
    print("RMSE:", round(rmse,4))
    print("R²  :", round(r2,4))

    return [name, mae, rmse, r2]

# =====================================================
# LINEAR REGRESSION
# =====================================================

lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_result = evaluate_model(
    lr_model,
    "Linear Regression"
)

# =====================================================
# RANDOM FOREST
# =====================================================

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=20,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_result = evaluate_model(
    rf_pipeline,
    "Random Forest"
)

# =====================================================
# XGBOOST
# =====================================================

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        )
    )
])

xgb_result = evaluate_model(
    xgb_pipeline,
    "XGBoost"
)
# Train model
xgb_pipeline.fit(
    X_train,
    y_train
)

# Predict test set
pred = xgb_pipeline.predict(
    X_test
)

prediction_df = pd.DataFrame({

    "Employee_Id":
        df.iloc[X_test.index]["Employee_Id"],

    "Year_Month":
        df.iloc[X_test.index]["Year_Month"],

    "Actual_Productivity":
        y_test.values,

    "Predicted_Productivity":
        pred

})

prediction_df.to_excel(
    "Prediction_Output.xlsx",
    index=False
)
from google.colab import files

files.download(
    "Prediction_Output.xlsx"
)
print("Prediction file exported.")


# =====================================================
# CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("\nCross Validation Scores")
print(cv_scores)

print(
    "\nAverage R²:",
    round(cv_scores.mean(),4)
)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

xgb_pipeline.fit(
    X_train,
    y_train
)

feature_names = (
    xgb_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = (
    xgb_pipeline
    .named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 20 Features")

print(
    importance_df.head(20)
)

# =====================================================
# MODEL COMPARISON
# =====================================================

comparison_df = pd.DataFrame(
    [
        lr_result,
        rf_result,
        xgb_result
    ],
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

print("\nModel Comparison")

print(comparison_df)

# =====================================================
# EXPORT
# =====================================================

comparison_df.to_excel(
    "Model_Comparison.xlsx",
    index=False
)

importance_df.to_excel(
    "Feature_Importance.xlsx",
    index=False
)

files.download(
    "Model_Comparison.xlsx"
)

files.download(
    "Feature_Importance.xlsx"
)

print("\nCompleted Successfully")

Saving Employee_Performance_Cleaned.xlsx to Employee_Performance_Cleaned (1).xlsx
Uploaded File: Employee_Performance_Cleaned (1).xlsx

After Source Deduplication
employees : (7500, 18)
stores : (150, 7)
performance : (236591, 13)
kpis : (236591, 9)
business : (16200, 9)

Business After Aggregation
(16200, 7)

Shape Before Dedup
(236591, 41)

Shape After Dedup
(236591, 41)

Top Correlations
Productivity_Index          1.000000
Performance_Rating          0.999980
Manager_Evaluation          0.826959
Monthly_Bonus               0.553722
Employee_Satisfaction       0.406879
Benefits_Cost               0.377085
Engagement_Index            0.347596
Training_Hours              0.239339
Overtime_Hours              0.172510
Age                         0.024258
Base_Salary_Annual          0.010717
City_Longitude              0.009883
Store_Location_Latitude     0.006472
Store_Location_Longitude    0.002254
On_Time_Delivery            0.001218
Name: Productivity_Index, dtype: float64

Removed 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Prediction file exported.

Cross Validation Scores
[0.76452954 0.76391788 0.75185244 0.75056302 0.76509313]

Average R²: 0.7592

Top 20 Features
                                  Feature  Importance
6                 num__Manager_Evaluation    0.839252
3                      num__Monthly_Bonus    0.057341
28         cat__Employment_Type_Full-time    0.018263
4              num__Employee_Satisfaction    0.015282
30          cat__Employment_Type_Seasonal    0.007532
2                   num__Absenteeism_Days    0.006170
29         cat__Employment_Type_Part-time    0.005854
27        cat__Employment_Type_Contractor    0.004560
0                     num__Training_Hours    0.003155
1                     num__Overtime_Hours    0.002269
5                   num__Engagement_Index    0.001840
25     cat__Department_Meat/Fish & Bakery    0.001813
20          cat__Department_Fresh Produce    0.001773
34               cat__Education_Level_PhD    0.001745
13               cat__Job_Level_Executive    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Completed Successfully


In [3]:
#Chạy SHAP MỨC 3
# =====================================================
# MACHINE LEARNING REGRESSION
# Predict Productivity Index
# Employee_Performance_Cleaned.xlsx
# =====================================================

!pip install xgboost openpyxl -q
!pip install shap -q

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

# =====================================================
# UPLOAD FILE
# =====================================================

uploaded = files.upload()

FILE_PATH = next(iter(uploaded))

print(f"Uploaded File: {FILE_PATH}")

# =====================================================
# LOAD DATA
# =====================================================

employees = pd.read_excel(
    FILE_PATH,
    sheet_name="employees"
)

stores = pd.read_excel(
    FILE_PATH,
    sheet_name="stores"
)

performance = pd.read_excel(
    FILE_PATH,
    sheet_name="monthly_performance"
)

kpis = pd.read_excel(
    FILE_PATH,
    sheet_name="role_kpis"
)

business = pd.read_excel(
    FILE_PATH,
    sheet_name="business_outcomes"
)

# =====================================================
# REMOVE DUPLICATES IN SOURCE TABLES
# =====================================================

employees = employees.drop_duplicates()
stores = stores.drop_duplicates()
performance = performance.drop_duplicates()
kpis = kpis.drop_duplicates()
business = business.drop_duplicates()

print("\nAfter Source Deduplication")

print("employees :", employees.shape)
print("stores :", stores.shape)
print("performance :", performance.shape)
print("kpis :", kpis.shape)
print("business :", business.shape)

# =====================================================
# PREVENT MANY TO MANY JOIN
# =====================================================

business = business.groupby(
    [
        "Store_Id",
        "Department",
        "Year_Month"
    ],
    as_index=False
).agg({

    "Customer_Satisfaction":"mean",
    "Nps_Score":"mean",
    "Waste_Percentage":"mean",
    "On_Time_Delivery":"mean"

})

print("\nBusiness After Aggregation")

print(business.shape)

# =====================================================
# MERGE DATA
# =====================================================

df = performance.merge(
    employees,
    on="Employee_Id",
    how="left"
)

df = df.merge(
    stores,
    on="Store_Id",
    how="left"
)

df = df.merge(
    kpis[
        [
            "Employee_Id",
            "Year_Month",
            "Productivity_Index"
        ]
    ],
    on=[
        "Employee_Id",
        "Year_Month"
    ],
    how="left"
)

df = df.merge(
    business,
    on=[
        "Store_Id",
        "Department",
        "Year_Month"
    ],
    how="left"
)

print("\nShape Before Dedup")
print(df.shape)

df = df.drop_duplicates()

print("\nShape After Dedup")
print(df.shape)

# =====================================================
# FEATURES
# =====================================================

features = [

    "Training_Hours",
    "Overtime_Hours",
    "Absenteeism_Days",
    "Monthly_Bonus",
    "Employee_Satisfaction",
    "Engagement_Index",
    "Manager_Evaluation",

    "Customer_Satisfaction",
    "Nps_Score",
    "Waste_Percentage",
    "On_Time_Delivery",

    "Job_Level",
    "Department",
    "Employment_Type",
    "Education_Level",
    "Store_Type"
]

target = "Productivity_Index"

# =====================================================
# NUMERIC
# =====================================================

numeric_features = [

    "Training_Hours",
    "Overtime_Hours",
    "Absenteeism_Days",
    "Monthly_Bonus",
    "Employee_Satisfaction",
    "Engagement_Index",
    "Manager_Evaluation",

    "Customer_Satisfaction",
    "Nps_Score",
    "Waste_Percentage",
    "On_Time_Delivery"
]

# =====================================================
# CATEGORICAL
# =====================================================

categorical_features = [

    "Job_Level",
    "Department",
    "Employment_Type",
    "Education_Level",
    "Store_Type"
]

# =====================================================
# REMOVE NULL TARGET
# =====================================================

df = df.dropna(subset=[target])

# =====================================================
# CORRELATION CHECK
# =====================================================

print("\nTop Correlations")

corr_check = (
    df.select_dtypes(include=np.number)
      .corr()["Productivity_Index"]
      .sort_values(ascending=False)
)

print(corr_check.head(15))

# =====================================================
# FINAL DEDUP
# =====================================================

before_rows = len(df)

df = df.drop_duplicates()

after_rows = len(df)

print(
    f"\nRemoved {before_rows-after_rows} duplicate rows"
)

print("Final Shape:", df.shape)

# =====================================================
# X / y
# =====================================================
print("\nCorrelation After Removing Leakage Variable")

corr_check = (
    df[
        numeric_features + [target]
    ]
    .corr()[target]
    .sort_values(ascending=False)
)

print(corr_check)

X = df[features]
y = df[target]

# =====================================================
# PREPROCESSING
# =====================================================

numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    [
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================================================
# EVALUATION FUNCTION
# =====================================================

def evaluate_model(model, name):

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    print("\n" + "="*50)
    print(name)
    print("="*50)

    print("MAE :", round(mae,4))
    print("RMSE:", round(rmse,4))
    print("R²  :", round(r2,4))

    return [name, mae, rmse, r2]

# =====================================================
# XGBOOST
# =====================================================

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        )
    )
])

xgb_result = evaluate_model(
    xgb_pipeline,
    "XGBoost"
)
# Train model
xgb_pipeline.fit(
    X_train,
    y_train
)

# Predict test set
pred = xgb_pipeline.predict(
    X_test
)

# =====================================================
# SHAP EXPLAINABLE AI
# =====================================================

print("\nRunning SHAP Analysis...")

# Lấy model XGBoost
xgb_model = xgb_pipeline.named_steps["model"]

# Transform dữ liệu giống lúc train
X_test_transformed = (
    xgb_pipeline
    .named_steps["preprocessor"]
    .transform(X_test)
)

# Lấy tên feature sau One-Hot Encoding
feature_names = (
    xgb_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("Total Features:", len(feature_names))

# SHAP Tree Explainer
explainer = shap.TreeExplainer(
    xgb_model
)

# SHAP Values
shap_values = explainer.shap_values(
    X_test_transformed
)

print("SHAP completed")

# =====================================================
# SHAP SUMMARY PLOT
# =====================================================

plt.figure(figsize=(12,8))

shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()

plt.savefig(
    "SHAP_Summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("SHAP Summary exported")

# =====================================================
# SHAP FEATURE IMPORTANCE TABLE
# =====================================================

importance_shap = np.abs(
    shap_values
).mean(axis=0)

shap_importance_df = pd.DataFrame({

    "Feature": feature_names,
    "SHAP_Importance": importance_shap

})

shap_importance_df = (
    shap_importance_df
    .sort_values(
        "SHAP_Importance",
        ascending=False
    )
)

print("\nTop 20 SHAP Features")

print(
    shap_importance_df.head(20)
)

shap_importance_df.to_excel(
    "SHAP_Feature_Importance.xlsx",
    index=False
)

print("SHAP Feature Importance exported")

prediction_df = pd.DataFrame({

    "Employee_Id":
        df.iloc[X_test.index]["Employee_Id"],

    "Year_Month":
        df.iloc[X_test.index]["Year_Month"],

    "Actual_Productivity":
        y_test.values,

    "Predicted_Productivity":
        pred

})

prediction_df.to_excel(
    "Prediction_Output.xlsx",
    index=False
)
from google.colab import files

files.download(
    "Prediction_Output.xlsx"
)
files.download(
    "SHAP_Summary.png"
)

files.download(
    "SHAP_Feature_Importance.xlsx"
)


Saving Employee_Performance_Cleaned.xlsx to Employee_Performance_Cleaned (2).xlsx
Uploaded File: Employee_Performance_Cleaned (2).xlsx

After Source Deduplication
employees : (7500, 18)
stores : (150, 7)
performance : (236591, 13)
kpis : (236591, 9)
business : (16200, 9)

Business After Aggregation
(16200, 7)

Shape Before Dedup
(236591, 41)

Shape After Dedup
(236591, 41)

Top Correlations
Productivity_Index          1.000000
Performance_Rating          0.999980
Manager_Evaluation          0.826959
Monthly_Bonus               0.553722
Employee_Satisfaction       0.406879
Benefits_Cost               0.377085
Engagement_Index            0.347596
Training_Hours              0.239339
Overtime_Hours              0.172510
Age                         0.024258
Base_Salary_Annual          0.010717
City_Longitude              0.009883
Store_Location_Latitude     0.006472
Store_Location_Longitude    0.002254
On_Time_Delivery            0.001218
Name: Productivity_Index, dtype: float64

Removed 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>